# Quantum Time Evolution in the XXZ Spin Chain: Exact, Trotterised, and Noisy Simulations

**PH10110 Quantum Computing Project**  
**Authors:** 
**Date:** 14 March 2026

This notebook is written as the final-report draft for the time-evolution project. It is intentionally notebook-first: the scientific narrative, mathematical structure, figure plan, and evidence trail are fixed here before the underlying software is treated as fully trusted. Any point that still requires code-level verification is marked explicitly rather than being silently presented as established fact.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'GroupProject' / 'results'

CASE_SPECS = [
    {'name': 'case_A_Jz_gt_1_all_down', 'label': 'Case A: $J_z = 1.5$, all spins down, single rotated site', 'L': 8, 'Jz': 1.5, 'boundary': 'open', 'init_pattern': 'all0', 'phi': np.pi / 3, 'rotate_site': 4},
    {'name': 'case_B_Jz_gt_1_all_up', 'label': 'Case B: $J_z = 1.5$, all spins up, single rotated site', 'L': 8, 'Jz': 1.5, 'boundary': 'open', 'init_pattern': 'all1', 'phi': np.pi / 3, 'rotate_site': 4},
    {'name': 'case_C_Jz_lt_minus1_alternating', 'label': 'Case C: $J_z = -1.5$, alternating initial state, single rotated site', 'L': 8, 'Jz': -1.5, 'boundary': 'open', 'init_pattern': 'alternating10', 'phi': np.pi / 3, 'rotate_site': 4},
]

REQUIRED_TAGS = [
    '[CODE ADDITION REQUIRED: Aer-based noisy evolution pipeline]',
    '[CODE VALIDATION REQUIRED: gate-angle sign convention for RXX/RYY/RZZ]',
    '[CODE VALIDATION REQUIRED: bit ordering consistency between classical tensors and Qiskit qubits]',
    '[CODE VALIDATION REQUIRED: observable extraction for noisy density-matrix simulations]',
    '[RESULT VALIDATION REQUIRED: full-resolution rerun for publication-quality figures]',
    '[RESULT VALIDATION REQUIRED: confirm whether symmetry observations persist after the final rerun]',
    '[MISSING INPUT: author names, contribution split, optional hardware data]',
]

def load_case_bundle(case_name: str) -> dict:
    bundle: dict[str, object] = {}
    metrics_path = RESULTS_DIR / f'{case_name}_metrics.json'
    data_path = RESULTS_DIR / f'{case_name}_data.npz'
    if metrics_path.exists():
        bundle['metrics'] = json.loads(metrics_path.read_text(encoding='utf-8'))
    if data_path.exists():
        with np.load(data_path) as raw:
            bundle['data'] = {key: raw[key] for key in raw.files}
    return bundle

RESULT_BUNDLES = {spec['name']: load_case_bundle(spec['name']) for spec in CASE_SPECS}

def show_markdown(text: str) -> None:
    display(Markdown(text))

def metrics_table() -> str:
    header = '| Case | Cached metrics available | Cached arrays available | Noiseless RMSE | Noisy RMSE | Qiskit arrays cached |\n|---|---:|---:|---:|---:|---:|\n'
    rows = []
    for spec in CASE_SPECS:
        bundle = RESULT_BUNDLES[spec['name']]
        metrics = bundle.get('metrics', {})
        data = bundle.get('data', {})
        rows.append(f"| {spec['label']} | {'yes' if 'metrics' in bundle else 'no'} | {'yes' if 'data' in bundle else 'no'} | {metrics.get('trajectory_vs_exact_rmse', 'n/a')} | {metrics.get('noisy_vs_exact_rmse', 'n/a')} | {'yes' if 'obs_qiskit' in data else 'no'} |")
    return header + '\n'.join(rows)

show_markdown('### Notebook Status\n\nThis draft reads cached artefacts from `GroupProject/results/` when they are available. Missing sections are left visible and are marked as technical work rather than being hidden.')
show_markdown(metrics_table())
show_markdown('### Explicit Validation Log\n\n' + '\n\n'.join(f'- {tag}' for tag in REQUIRED_TAGS))


## 2. Executive Overview / Abstract

This report studies real-time dynamics in a one-dimensional XXZ spin chain with nearest-neighbour couplings. The target problem is to prepare one of several prescribed product states, rotate a single spin into the equatorial plane, evolve the system under the XXZ Hamiltonian, and track the local expectation values $\langle X_i(t)\rangle$, $\langle Y_i(t)\rangle$, and $\langle Z_i(t)\rangle$ across the full chain. The project therefore combines a well-defined many-body physics problem with a natural quantum-computing task: the digital approximation of time evolution by a sequence of local gates.

The report is organised around three layers of evidence. First, exact classical simulation provides the baseline against which every approximate method is judged. Second, Trotterised circuit evolution is used to test whether a digital quantum representation reproduces the intended dynamics with controlled error. Third, noisy simulation is introduced to assess how rapidly physically meaningful structure is degraded once imperfections are included. At the system sizes considered here, this project does not establish quantum advantage. Its value is instead methodological: it shows how exact benchmarks, product-formula time evolution, and noise analysis can be combined into a reproducible and critical study of quantum simulation.

## 3. Introduction

The simulation of quantum many-body dynamics is one of the clearest long-term motivations for quantum computing. The central challenge is not merely the storage of a large wavefunction, but the faithful propagation of that wavefunction under an interacting Hamiltonian while preserving physically relevant observables. In this project the testbed is the XXZ spin chain, a standard model that is simple enough to formulate cleanly and rich enough to exhibit non-trivial dynamics, symmetry constraints, and parameter-dependent behaviour.

For the small systems used in this report, exact classical simulation remains both feasible and essential. This is not a weakness of the project design; it is a necessary scientific control. Exact statevector evolution provides the benchmark that allows approximate digital evolution to be evaluated honestly. The role of the quantum-computing component is therefore not to outperform classical computation at $L=8$, but to implement the same physics through a circuit model, quantify the Trotter error introduced by digitisation, and examine how far those conclusions survive once noise is introduced.

The report adopts a deliberately cautious position on usefulness. For the present parameter regime, exact classical computation is cheaper, cleaner, and more reliable than a quantum workflow. Nevertheless, quantum circuits remain relevant because they provide the formal route by which Hamiltonian dynamics would be implemented on gate-based hardware. Studying that route in a controlled small-scale setting reveals which aspects of the dynamics are robust, which are artefacts of coarse Trotterisation, and which are rapidly destroyed by noise or circuit depth.

## 4. Problem Formulation and Theoretical Background

The project concerns the one-dimensional XXZ Hamiltonian with open boundary conditions,

\[
H = -\sum_{i=0}^{L-2} \left(X_i X_{i+1} + Y_i Y_{i+1} + J_z Z_i Z_{i+1}\right),
\]

where $X_i$, $Y_i$, and $Z_i$ denote Pauli operators acting on lattice site $i$, and $J_z$ is the anisotropy parameter. The negative overall sign is part of the project definition and must therefore be tracked consistently throughout both the classical and circuit formulations. The chain is initialised in one of three regimes prescribed by the project brief: the all-down product state, the all-up product state, or an alternating $|1010\ldots\rangle$ configuration. A single lattice site, chosen here to be the middle site unless stated otherwise, is then rotated into the equatorial plane by a local phase-dependent unitary.

The observables of interest are the local magnetisation components,

\[
\langle X_i(t)\rangle, \qquad \langle Y_i(t)\rangle, \qquad \langle Z_i(t)\rangle,
\]

measured at every site and every sampled time. This generates a spacetime picture of how a local perturbation spreads through the chain. Physically, the anisotropy controls the relative weight of longitudinal and transverse couplings, so it changes both the stability of the initial configuration and the character of the subsequent propagation. The all-up and all-down states also provide a useful symmetry check, since many features should be related by global spin inversion if the implementation conventions are consistent.

**Technical status.**  
[TECHNICAL VALIDATION NEEDED: sign convention / qubit ordering / site indexing]

This tag is retained intentionally. The notebook narrative fixes the target mathematical problem first; any mismatch between that target and the present software implementation must be checked explicitly rather than assumed away.

In [ ]:
table_lines = [
    '| Quantity | Case A | Case B | Case C |',
    '|---|---:|---:|---:|',
    f"| Chain length $L$ | {CASE_SPECS[0]['L']} | {CASE_SPECS[1]['L']} | {CASE_SPECS[2]['L']} |",
    f"| Anisotropy $J_z$ | {CASE_SPECS[0]['Jz']} | {CASE_SPECS[1]['Jz']} | {CASE_SPECS[2]['Jz']} |",
    f"| Boundary | {CASE_SPECS[0]['boundary']} | {CASE_SPECS[1]['boundary']} | {CASE_SPECS[2]['boundary']} |",
    f"| Initial pattern | {CASE_SPECS[0]['init_pattern']} | {CASE_SPECS[1]['init_pattern']} | {CASE_SPECS[2]['init_pattern']} |",
    '| Rotation phase $\\phi$ | $\\pi/3$ | $\\pi/3$ | $\\pi/3$ |',
    f"| Rotated site | {CASE_SPECS[0]['rotate_site']} | {CASE_SPECS[1]['rotate_site']} | {CASE_SPECS[2]['rotate_site']} |",
]
show_markdown('### Table 1. Working Case Definitions\n\n' + '\n'.join(table_lines))
show_markdown('The values in Table 1 are treated as the intended report parameters. If the underlying scripts are changed later, the notebook should be updated here first so that the report remains the source of truth for the scientific specification.')


## 5. Methods

### 5.1 Classical Formulation

The most reliable baseline for this project is exact statevector evolution. Once the Hamiltonian matrix has been assembled, the wavefunction can be propagated according to $|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$. In practice, sparse-matrix methods such as `expm_multiply` are preferable to constructing the full exponential explicitly, because they apply the evolution operator to the state without materialising the entire dense matrix exponential. For the present system size this remains computationally manageable, and it provides a benchmark that is effectively exact on the scale relevant to the later comparisons.

The key classical limitation is the exponential growth of Hilbert-space dimension. A chain of length $L$ requires a statevector of size $2^L$, and the matrix representation of the Hamiltonian grows accordingly. This makes exact evolution an excellent benchmark for small systems but not a scalable general solution. That scaling argument is important to the report because it explains why quantum circuits are of conceptual interest even when classical methods are still superior at the present problem size.

### 5.2 Quantum Formulation

The quantum-circuit approach approximates the exact propagator by a product formula. Writing the Hamiltonian as a sum of local two-site terms $H = \sum_b h_b$, one replaces the full propagator over a short interval $\Delta t$ by an ordered product of local exponentials. The first-order Suzuki-Trotter formula is

\[
e^{-iH\Delta t} = e^{-i\sum_b h_b\Delta t} \approx \prod_b e^{-ih_b\Delta t} + O(\Delta t^2),
\]

while the symmetric second-order formula is

\[
e^{-iH\Delta t} \approx \prod_b e^{-ih_b\Delta t/2} \prod_{b\,\text{in reverse}} e^{-ih_b\Delta t/2} + O(\Delta t^3).
\]

This decomposition is natural for the XXZ model because each bond contributes only a two-site interaction. The trade-off is equally natural: smaller time steps reduce Trotter error but require more circuit layers, so the algorithm becomes deeper and more exposed to noise.

### 5.3 Circuit Design

For each bond $(i,j)$ the local propagator is implemented through a two-qubit unitary corresponding to the XX, YY, and ZZ interactions. In a gate-native implementation this is naturally expressed in terms of `RXX`, `RYY`, and `RZZ` rotations with angles chosen to reproduce the sign convention and coupling strength of the Hamiltonian. Initial-state preparation is comparatively straightforward because the prescribed states are product states, after which only a single-site rotation is needed to place one spin in the equatorial plane. Repeating the Trotter layer over a time grid then yields a digital approximation to the full dynamics.

**Technical status.**  
[TECHNICAL VALIDATION NEEDED: exact gate-angle convention in current circuit implementation]

This point must be checked carefully in the final implementation, because a sign error or qubit-ordering mismatch would not merely perturb the result; it would change the simulated Hamiltonian.

### 5.4 Implementation Details

The report is organised around three fixed cases, listed in Table 1, and four computational layers: exact classical evolution, noiseless Trotterised evolution, noisy circuit simulation, and optional hardware execution. The notebook is designed so that the reporting layer can survive code changes beneath it. Wherever a parameter, convention, or output path is not yet fully stable, the uncertainty is made explicit in the text rather than being absorbed into the prose as if it were settled.

In [ ]:
def plot_spacetime_triptych(obs: np.ndarray, times: np.ndarray, title: str):
    labels = [r'$\\langle X_i(t)\\rangle$', r'$\\langle Y_i(t)\\rangle$', r'$\\langle Z_i(t)\\rangle$']
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.8), constrained_layout=True)
    for idx, ax in enumerate(axes):
        im = ax.imshow(obs[:, :, idx], origin='lower', aspect='auto', cmap='coolwarm', vmin=-1.0, vmax=1.0, extent=[0, obs.shape[1] - 1, times[0], times[-1]])
        ax.set_title(labels[idx])
        ax.set_xlabel('site index $i$')
        ax.set_ylabel('time $t$')
        fig.colorbar(im, ax=ax, shrink=0.85)
    fig.suptitle(title)
    return fig

def plot_error_scaling(metrics: dict, title: str):
    steps = np.asarray(metrics['error_steps'], dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
    axes[0].loglog(steps, metrics['infidelity_order1'], 'o-', label='1st order')
    axes[0].loglog(steps, metrics['infidelity_order2'], 's-', label='2nd order')
    axes[0].set_xlabel('number of Trotter steps')
    axes[0].set_ylabel('final-state infidelity')
    axes[0].set_title('State error')
    axes[0].grid(True, which='both', alpha=0.25)
    axes[0].legend()
    axes[1].loglog(steps, metrics['rmse_order1'], 'o-', label='1st order')
    axes[1].loglog(steps, metrics['rmse_order2'], 's-', label='2nd order')
    axes[1].set_xlabel('number of Trotter steps')
    axes[1].set_ylabel('RMSE of local observables')
    axes[1].set_title('Observable error')
    axes[1].grid(True, which='both', alpha=0.25)
    axes[1].legend()
    fig.suptitle(title)
    return fig

def plot_fft_pair(z_data: np.ndarray, times: np.ndarray, title: str):
    raw_fft = np.abs(np.fft.fftshift(np.fft.fft2(z_data)))
    centred_fft = np.abs(np.fft.fftshift(np.fft.fft2(z_data - z_data.mean())))
    dt = float(times[1] - times[0]) if len(times) > 1 else 1.0
    omega = np.fft.fftshift(np.fft.fftfreq(z_data.shape[0], d=dt)) * 2.0 * np.pi
    k = np.fft.fftshift(np.fft.fftfreq(z_data.shape[1], d=1.0)) * 2.0 * np.pi
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0), constrained_layout=True)
    for ax, arr, subtitle in zip(axes, [raw_fft, centred_fft], ['raw FFT', 'mean-subtracted FFT']):
        im = ax.imshow(arr, origin='lower', aspect='auto', cmap='magma', extent=[k[0], k[-1], omega[0], omega[-1]])
        ax.set_xlabel('wave number $k$')
        ax.set_ylabel('angular frequency $\\omega$')
        ax.set_title(subtitle)
        fig.colorbar(im, ax=ax, shrink=0.85)
    fig.suptitle(title)
    return fig


## 6. Classical Benchmarks

This section is intentionally treated as the most trustworthy part of the report. Exact classical evolution provides the reference dynamics against which every later approximation is judged. For that reason the exact spacetime plots are not merely illustrative; they define the standard that Trotterised and noisy simulations must reproduce if they are to be considered physically meaningful.

The most useful interpretation is not just that the perturbation spreads, but how it spreads. The observed structure can depend on the chosen initial sector, the anisotropy, the finite open chain, and conserved quantities such as total magnetisation in the noiseless setting. In a final polished version, this section should therefore combine the figures below with explicit comments on propagation fronts, boundary reflections, and the degree to which the local excitation remains constrained by the background state.

**Result status.**  
[RESULT VALIDATION REQUIRED: full-resolution rerun for publication-quality figures]

In [ ]:
for spec in CASE_SPECS:
    bundle = RESULT_BUNDLES[spec['name']]
    if 'data' not in bundle:
        show_markdown(f"**{spec['label']}.** Cached exact data are not currently available in `GroupProject/results/`.")
        continue
    data = bundle['data']
    fig = plot_spacetime_triptych(data['obs_exact'], data['times'], f"Exact dynamics: {spec['label']}")
    plt.show()
    total_z = data['obs_exact'][:, :, 2].sum(axis=1)
    show_markdown(f"*Figure: exact spacetime observables for {spec['label']}.* In the current cached dataset, the total longitudinal magnetisation varies between {total_z.min():.6f} and {total_z.max():.6f}. This is a useful sanity check for the noiseless benchmark, although it should be re-evaluated after the final high-resolution rerun.")


## 7. Exact Quantum Simulation Results

The purpose of this section is not to argue that the circuit model is superior to the exact classical benchmark. Its purpose is narrower and more important: to determine whether the digital circuit construction reproduces the intended XXZ dynamics when evaluated in an exact statevector setting. If the circuit conventions are correct, then the noiseless Trotterised circuit should converge towards the exact benchmark as the time step is refined, and an independent implementation should agree with the in-house Trotter calculation to numerical precision.

At the present draft stage, the notebook distinguishes clearly between what is already cached and what still needs to be regenerated. The existing quick-run artefacts are sufficient to structure the report and demonstrate the expected comparison logic, but they should not yet be treated as the final figures for submission.

**Technical status.**  
If the current circuit convention is confirmed, the exact circuit simulation should reproduce the benchmark to within numerical precision.  
[CODE VALIDATION REQUIRED: bit ordering consistency between classical tensor ordering and Qiskit qubit ordering]

In [ ]:
rows = ['| Case | Noiseless Trotter vs exact RMSE | Noisy vs exact RMSE | Cached Qiskit data |', '|---|---:|---:|---:|']
for spec in CASE_SPECS:
    bundle = RESULT_BUNDLES[spec['name']]
    metrics = bundle.get('metrics', {})
    data = bundle.get('data', {})
    rows.append(f"| {spec['label']} | {metrics.get('trajectory_vs_exact_rmse', 'n/a')} | {metrics.get('noisy_vs_exact_rmse', 'n/a')} | {'yes' if 'obs_qiskit' in data else 'no'} |")
show_markdown('### Table 2. Cached Error Summary\n\n' + '\n'.join(rows))
show_markdown('The current cache captures the exact benchmark and the in-house Trotter comparison. In the present repository snapshot, Qiskit comparison arrays are not yet stored alongside the quick-run data, so this section remains structurally complete but evidentially provisional.')
for spec in CASE_SPECS:
    bundle = RESULT_BUNDLES[spec['name']]
    metrics = bundle.get('metrics')
    if metrics is None:
        continue
    fig = plot_error_scaling(metrics, f"Trotter error scaling: {spec['label']}")
    plt.show()
    show_markdown(f"*Figure: Trotter error scaling for {spec['label']}.* The current quick-run data already show the expected qualitative pattern: the second-order product formula yields substantially lower error than the first-order construction at fixed step count. The final report should restate this with full-resolution data and, where useful, a fitted scaling exponent.")


## 8. Noisy Simulation Results

Noisy simulation is needed for a different reason from exact benchmarking. The exact benchmark establishes correctness; the noisy simulation tests fragility. A Trotter circuit can agree with the target Hamiltonian in principle and still become physically uninformative once finite gate error, decoherence, or readout distortion are introduced. The central question is therefore not whether the noisy output still resembles the exact plot by eye, but which observables remain quantitatively usable and which physical signatures are washed out first.

The intended final version of this report will use a Qiskit Aer gate-level noise model as the primary noisy backend. That is the correct standard for a report of this type because it keeps the circuit structure visible while introducing a simulator explicitly designed for noisy quantum computation. If Aer is not yet integrated at the implementation stage, the notebook permits a clearly labelled fallback subsection using a simplified stochastic local-Pauli model. That fallback must never be described as hardware-calibrated noise.

**Technical status.**  
[CODE ADDITION REQUIRED: Aer-based noisy evolution pipeline]  
[CODE VALIDATION REQUIRED: observable extraction for noisy density-matrix simulations]

In [ ]:
for spec in CASE_SPECS:
    bundle = RESULT_BUNDLES[spec['name']]
    if 'data' not in bundle:
        continue
    data = bundle['data']
    fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.8), constrained_layout=True)
    center = data['obs_exact'].shape[1] // 2
    axes[0].plot(data['times'], data['obs_exact'][:, center, 2], label='exact', linewidth=2)
    axes[0].plot(data['times'], data['obs_trotter'][:, center, 2], '--', label='noiseless Trotter')
    axes[0].plot(data['times'], data['obs_noisy'][:, center, 2], ':', label='cached noisy model')
    axes[0].set_xlabel('time $t$')
    axes[0].set_ylabel(r'centre-site $\\langle Z(t)\\rangle$')
    axes[0].set_title('Representative local observable')
    axes[0].legend()
    total_z_exact = data['obs_exact'][:, :, 2].sum(axis=1)
    total_z_noisy = data['obs_noisy'][:, :, 2].sum(axis=1)
    axes[1].plot(data['times'], total_z_exact, label='exact total $Z$')
    axes[1].plot(data['times'], total_z_noisy, label='cached noisy total $Z$')
    axes[1].set_xlabel('time $t$')
    axes[1].set_ylabel(r'$\\sum_i \\langle Z_i(t)\\rangle$')
    axes[1].set_title('Conserved quantity under noise')
    axes[1].legend()
    plt.show()
    show_markdown(f"*Figure: noisy comparison for {spec['label']}.* The current cached noisy trajectories are exploratory only. They are still useful for report drafting because they show the type of comparison that matters: local signal distortion and drift in quantities that are conserved in the noiseless benchmark. These plots must be replaced or supplemented once the Aer backend is implemented.")


## 9. Optional Hardware Results

Real-device runs are optional for this project and are therefore not treated as mandatory evidence. Their value would lie in testing whether a deliberately simplified version of the circuit still produces behaviour consistent with both the exact benchmark and the noisy simulator. That comparison is scientifically useful only if the hardware results are presented honestly, with the relevant job metadata, transpilation choices, and limitations stated clearly.

At the time of writing, no verified hardware dataset has been inserted into this notebook. The section is nevertheless retained so that the final submission keeps the correct academic structure and so that any later hardware data can be integrated without disturbing the report narrative.

[MISSING INPUT: IBM Quantum hardware job IDs / exported counts / figures]

In the absence of such data, the correct statement is simply that real-device execution is not included in the present report.

In [ ]:
# Optional hardware template: leave commented out in the submitted Run All workflow.
#
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService()
# backend = service.backend('ibm_backend_name')
#
# [MISSING INPUT: hardware circuit definition, job IDs, counts, mitigation strategy]
#
# The submitted notebook should keep hardware cells disabled unless they are fully
# documented and known not to break top-to-bottom execution.


## 10. Discussion

The central physical question is how a local perturbation propagates in the XXZ chain under different background states and anisotropies. In the final report this should be answered comparatively rather than case by case in isolation. The all-down and all-up configurations test whether the implementation respects the expected symmetry between opposite polarisation sectors, while the alternating state probes a qualitatively different background in which the perturbation is embedded in a patterned configuration rather than a uniform one.

From an algorithmic perspective, the most important conclusion is not that Trotterisation works in some vague sense, but that its failure mode is understandable. Decreasing the step size improves the approximation, especially for the second-order formula, but it also increases circuit depth. This is the core design tension for digital time evolution on gate-based hardware: the improvement in discretisation error is purchased with a greater exposure to hardware noise.

Noise then changes the interpretation of the dynamics. In the exact and noiseless settings, conserved quantities and symmetry relations provide useful internal checks. Under noisy evolution, these checks become diagnostics for degradation. A drift in total magnetisation, for example, is not merely a numerical nuisance; it indicates that the simulator no longer preserves the structure of the target problem. The practical value of the noisy comparison therefore lies in identifying which qualitative patterns survive and which do not.

The Fourier analysis should also be interpreted carefully. A raw two-dimensional FFT of $\langle Z_i(t)\rangle$ can be dominated by the zero-frequency, zero-wave-number component if the data carry a large static offset. That feature is mathematically real but can obscure the propagating content of interest. For this reason the notebook includes a mean-subtracted FFT as a supplementary diagnostic. This does not replace the raw spectrum; it clarifies which structures are associated with dynamical modulation rather than a static background.

In [ ]:
representative_case = CASE_SPECS[2]['name']
bundle = RESULT_BUNDLES.get(representative_case, {})
if 'data' in bundle:
    data = bundle['data']
    fig = plot_fft_pair(data['obs_exact'][:, :, 2], data['times'], 'FFT diagnostics for the exact $Z$ observable (representative case)')
    plt.show()
    show_markdown('*Figure: raw and mean-subtracted FFT of the exact longitudinal magnetisation.* The raw FFT preserves the full signal, including the static offset. The mean-subtracted FFT is shown only as an interpretive aid, so that dispersive or oscillatory structure is not visually overwhelmed by the DC component.')
else:
    show_markdown('Cached exact data for the representative FFT analysis are not currently available.')


## 11. Conclusion

This project is best understood as a controlled study of digital quantum simulation rather than a claim of practical quantum advantage. Exact classical evolution remains indispensable because it supplies the benchmark against which every later layer of approximation is judged. Within that benchmarked setting, Trotterised time evolution provides a clean and physically interpretable route from the XXZ Hamiltonian to a gate-based circuit representation.

The main limitation is equally clear. Better Trotter accuracy requires deeper circuits, and deeper circuits are precisely what make noisy and hardware-level implementations fragile. A result that is compelling in noiseless statevector simulation may become significantly degraded once realistic imperfections are introduced. For that reason the final report must treat exact, noiseless-circuit, noisy-simulation, and optional hardware evidence as distinct levels of reliability rather than merging them into a single undifferentiated story.

The most realistic next steps are therefore targeted rather than grandiose: verify the circuit conventions rigorously, regenerate full-resolution figures, add a proper Aer noisy backend, and, if hardware data are obtained, reduce the execution target to a set of observables and circuit depths that can be defended quantitatively. Those improvements would strengthen the report without requiring any unsupported claim about near-term quantum superiority.

## 12. Contribution Statement

The final manuscript was assembled jointly. Specific attribution will be inserted once the team has finalised the division of work.

A recommended final format is to separate contributions into the following categories: background and theory, classical benchmarking, circuit implementation, noisy simulation, hardware execution, and report editing. If figures from different team members are merged into the final notebook, the caption or surrounding text should identify which member produced which dataset.

## 13. References

[1] S. J. Thomson, *PH10110 Quantum Computing Project: Time Evolution*, project brief, February 2026.

[2] Qiskit documentation, IBM Quantum / Qiskit SDK documentation for circuit construction, statevector simulation, and noisy simulation workflows.

[3] M. Suzuki, "General theory of higher-order decomposition of exponential operators and symplectic integrators," *Physics Letters A* **165**, 387-395 (1992).

[4] J. Trotter, "On the product of semi-groups of operators," *Proceedings of the American Mathematical Society* **10**, 545-551 (1959).

[5] A standard reference on quantum spin chains or many-body quantum mechanics to be inserted in the final version, for example a textbook or lecture notes covering Heisenberg/XXZ models.

**Reference status.**  
[MISSING INPUT: final reference list formatted consistently after the team agrees the bibliography standard]